# Meta-Atom Library Optimization — Focusing with Interpolated Unit Cells

Notebook version of `scripts/metaatom_optimization_example.py`.  This example is closely
related to `examples/lens_optimization_example.ipynb`: we still optimize a device to focus
a plane wave at a target pixel after propagation.

The key difference is that the trainable surface is **not** an unconstrained phase mask.
Instead, each pixel is parameterized by a **meta-atom geometry** (here a square-pillar side
length), and the complex transmission is obtained from a precomputed library using
`MetaAtomInterpolationLayer`.

## 0  Imports

As in the lens optimization notebook, we use JAX for differentiation and Optax for
optimization.  The new pieces are:

- `MetaAtomLibrary` for the tabulated unit-cell response, and
- `MetaAtomInterpolationLayer` for converting trainable geometry parameters into a complex
  transmission map at the chosen wavelength.


In [ ]:
from __future__ import annotations

from pathlib import Path

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax

from fouriax.optics import (
    Field,
    Grid,
    MetaAtomInterpolationLayer,
    MetaAtomLibrary,
    OpticalModule,
    PhaseMask,
    Spectrum,
    plan_propagation,
)
from fouriax.optim import optimize_optical_module

# NOTEBOOK_REPO_ROOT_SETUP
import os
from pathlib import Path as _Path
%matplotlib inline

def _find_repo_root(start: _Path) -> _Path:
    for candidate in (start, *start.parents):
        if (
            (candidate / "src" / "fouriax").exists()
            and (candidate / "README.md").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root from current working directory. "
        "Expected to find src/fouriax and README.md in an ancestor."
    )

REPO_ROOT = _find_repo_root(_Path.cwd())
if _Path.cwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)


## 1  Paths and Parameters


In [ ]:
NPZ_PATH = Path("../../data/meta_atoms/square_pillar_0p7um_cell_sweep_results.npz")
ARTIFACTS_DIR = Path("artifacts")
PLOT_PATH = ARTIFACTS_DIR / "metaatom_opt_overview.png"
SUMMARY_PATH = ARTIFACTS_DIR / "metaatom_opt_summary.json"

SPEED_OF_LIGHT_M_PER_S = 299_792_458.0
SEED = 0
GRID_N = 64
GRID_DX_UM = 0.7
SELECTED_WAVELENGTH_UM = 1.30
DISTANCE_UM = 100.0
LR = 0.1
STEPS = 180


## 2 Helper Functions

The sweep file stores transmission magnitude and phase over a grid of:

- optical frequencies (or wavelengths), and
- geometric parameters (square-pillar side length).

The helper converts these tabulated values into a complex transmission response

$$
t(g, \lambda) = a(g, \lambda)\,e^{i\phi(g, \lambda)},
$$

and packages the result into a `MetaAtomLibrary` object.


In [ ]:
def load_square_pillar_library(npz_path: Path) -> MetaAtomLibrary:
    """Load LUT from NPZ keys: freqs [Hz], side_lengths [m], trans, phase."""
    with np.load(npz_path) as data:
        freqs_hz = np.asarray(data["freqs"], dtype=np.float64).reshape(-1)
        side_lengths_m = np.asarray(data["side_lengths"], dtype=np.float64).reshape(-1)
        trans = np.asarray(data["trans"], dtype=np.float64)
        phase = np.asarray(data["phase"], dtype=np.float64)

    wavelengths_um = (SPEED_OF_LIGHT_M_PER_S / freqs_hz) * 1e6
    side_lengths_um = side_lengths_m * 1e6

    wav_order = np.argsort(wavelengths_um)
    side_order = np.argsort(side_lengths_um)

    wavelengths_um = wavelengths_um[wav_order]
    side_lengths_um = side_lengths_um[side_order]

    trans = trans[side_order, :][:, wav_order]
    phase = phase[side_order, :][:, wav_order]

    transmission_complex = trans.T * np.exp(1j * phase.T)

    return MetaAtomLibrary.from_complex(
        wavelengths_um=jnp.asarray(wavelengths_um, dtype=jnp.float32),
        parameter_axes=(jnp.asarray(side_lengths_um, dtype=jnp.float32),),
        transmission_complex=jnp.asarray(transmission_complex),
    )


## 3  Setup

As in `examples/lens_optimization_example.ipynb`, we choose a grid, a propagation distance,
and a center pixel target.  The main extra step here is selecting a wavelength from the
meta-atom library (nearest available sample to `selected_wavelength_um`).

We also extract the valid geometry interval from the library axis and define
`min_bounds`/`max_bounds`, which constrain the interpolated meta-atom parameters during
optimization.


In [ ]:
library = load_square_pillar_library(NPZ_PATH)

nearest_idx = int(jnp.argmin(jnp.abs(library.wavelengths_um - SELECTED_WAVELENGTH_UM)))
wavelength_um = float(library.wavelengths_um[nearest_idx])
grid = Grid.from_extent(nx=GRID_N, ny=GRID_N, dx_um=GRID_DX_UM, dy_um=GRID_DX_UM)
spectrum = Spectrum.from_scalar(wavelength_um)
field_in = Field.plane_wave(grid=grid, spectrum=spectrum)

target_xy = (grid.nx // 2, grid.ny // 2)
propagator = plan_propagation(
    mode="auto",
    grid=grid,
    spectrum=spectrum,
    distance_um=DISTANCE_UM,
)

side_axis = library.parameter_axes[0]
min_bounds = jnp.array([side_axis[0]], dtype=jnp.float32)
max_bounds = jnp.array([side_axis[-1]], dtype=jnp.float32)

def build_module(raw_params: jnp.ndarray) -> OpticalModule:
    return OpticalModule(
        layers=(
            MetaAtomInterpolationLayer(
                library=library,
                raw_geometry_params=raw_params,
                min_geometry_params=min_bounds,
                max_geometry_params=max_bounds,
            ),
            propagator,
        )
    )


## 5  Loss Function and Optimization

### Bounded geometry parameterization

The trainable tensor `raw_params` is unconstrained, but the meta-atom layer maps it into the
library's valid geometry range.  Conceptually, each pixel parameter is bounded as

$$
g(x,y) \in [g_{\min}, g_{\max}],
$$

via a smooth transform (implemented inside `MetaAtomInterpolationLayer`).  The layer then
interpolates the complex transmission $t(g,\lambda)$ from the LUT at the selected
wavelength.

### Focusing objective (same idea as lens optimization)

Like the lens notebook, we maximize on-axis focusing at `target_xy`.  The propagated
intensity is

$$
I(x,y) = |E_{\mathrm{out}}(x,y)|^2,
$$

and this notebook uses the scalar objective

$$
\mathcal{L} = -I(x_0, y_0),
$$

where $(x_0, y_0)$ is the target pixel.  Minimizing $\mathcal{L}$ is therefore equivalent
to maximizing center intensity.

We initialize `raw_params` with small random values and set up Adam for optimization.


In [ ]:
def loss_fn(raw_params: jnp.ndarray) -> jnp.ndarray:
    module = build_module(raw_params)
    intensity = module.forward(field_in).intensity()
    center = intensity[0, target_xy[1], target_xy[0]]
    return -center

key = jax.random.PRNGKey(SEED)
raw_params = 0.1 * jax.random.normal(key, (grid.ny, grid.nx), dtype=jnp.float32)
optimizer = optax.adam(learning_rate=LR)

result = optimize_optical_module(
    init_params=raw_params,
    build_module=build_module,
    loss_fn=loss_fn,
    optimizer=optimizer,
    steps=STEPS,
    log_every=30,
)


## 7  Evaluation

After optimization, we evaluate the final meta-atom device and compare it to the same
hyperbolic-phase focusing reference used in the lens notebook:

$$
\phi_{\mathrm{ref}}(x,y) = -k\left(\sqrt{x^2+y^2+f^2}-f\right),
\qquad k = \frac{2\pi}{\lambda}.
$$

This reference is not subject to the meta-atom library constraints, so it serves as an
idealized baseline for profile comparison.

We also inspect the learned bounded geometry map (`optimized_side_map`) and query the
library to report the local complex transmission at the center pixel.


In [ ]:
final_intensity = np.asarray(result.best_module.forward(field_in).intensity())
optimized_profile = final_intensity[0, target_xy[1], :]

# Reference: phase profile for ideal spherical wavefront convergence at distance_um.
x_um, y_um = grid.spatial_grid()
k = 2.0 * jnp.pi / wavelength_um
hyperbolic_phase = -k * (
    jnp.sqrt(x_um * x_um + y_um * y_um + DISTANCE_UM**2) - DISTANCE_UM
)
reference_module = OpticalModule(
    layers=(
        PhaseMask(phase_map_rad=hyperbolic_phase[None, :, :]),
        propagator,
    )
)

reference_intensity = np.asarray(reference_module.forward(field_in).intensity())
reference_profile = reference_intensity[0, target_xy[1], :]

final_layer = MetaAtomInterpolationLayer(
    library=library,
    raw_geometry_params=result.best_params,
    min_geometry_params=min_bounds,
    max_geometry_params=max_bounds,
)
optimized_side_map = np.asarray(final_layer.bounded_geometry_params()[0])


## 8  Plot Results

The final section generates an overview figure with:

- optimization loss history,
- optimized side-length map,
- 2D focal intensity, and
- center-row intensity profile vs the hyperbolic reference.

These outputs make it easy to compare this **library-constrained** design workflow against
the unconstrained phase-mask lens optimization notebook.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.0))

axes[0, 0].plot(result.history)
axes[0, 0].set_title("Loss History")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(alpha=0.3)

side_im = axes[0, 1].imshow(optimized_side_map, cmap="viridis")
axes[0, 1].set_title("Optimized Meta-Atom Side-Lengths")
axes[0, 1].set_xticks([])
axes[0, 1].set_yticks([])
plt.colorbar(side_im, ax=axes[0, 1], fraction=0.046, pad=0.04)

focus_im = axes[1, 0].imshow(final_intensity[0], cmap="inferno")
axes[1, 0].set_title("Optimized 2D Focal Spot")
axes[1, 0].set_xticks([])
axes[1, 0].set_yticks([])
plt.colorbar(focus_im, ax=axes[1, 0], fraction=0.046, pad=0.04)

axes[1, 1].plot(optimized_profile, label="Optimized")
axes[1, 1].plot(reference_profile, label="Hyperbolic-phase reference", linestyle="--")
axes[1, 1].set_title("Center Row Profile")
axes[1, 1].set_xlabel("x pixel")
axes[1, 1].grid(alpha=0.3)
axes[1, 1].legend()

fig.tight_layout()
fig.savefig(PLOT_PATH, dpi=150)
plt.show()
print(f"saved: {PLOT_PATH}")
